# 03. Оценка модели: устойчивость и LLM-as-a-Judge

Три уровня проверки уже обученной локальной модели (`qwen-reviewer:latest` в Ollama):

| Блок | Что измеряем | Артефакт |
|---|---|---|
| Smoke-тест | модель отвечает валидным JSON по заданной схеме | вывод ячейки |
| Количественный тест | JSON validity + языковая консистентность (RU, без языкового сдвига) | `reports/batch_reviews.csv` |
| LLM-as-a-Judge | точность / полнота / логика рецензии по версии DeepSeek-V3 | `reports/evaluation_results.jsonl` |

Требования: запущенная Ollama с моделью `qwen-reviewer:latest` (`modelfiles/Modelfile`),
для блока 3 — `CHUTES_API_TOKEN` в `.env`.

Почему это важно: модель после LoRA на статьях с английским abstract склонна «сваливаться»
в английский, а её вердикт может не соответствовать выставленным оценкам. Оба дефекта
измеряются здесь и лечатся в `src/ollama_client.py` (schema-constrained decoding +
«якорь свежести» на русском) и `src/censor.py` (детерминированный вердикт).

In [ ]:
# --- Repo-relative пути: ноутбуки лежат в notebooks/, данные — в корне репозитория ---
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("Рабочий каталог:", REPO_ROOT)

## 1. Smoke-тест: рецензия на одну статью (schema-constrained output)

In [ ]:
"""Блок 1. Smoke-тест: одна статья, один вызов, проверка схемы ответа."""
import json

from src.config import REPO_ROOT
from src.ollama_client import generate_review

SAMPLE_ARTICLE = REPO_ROOT / "examples" / "sample_article.txt"

article_text = SAMPLE_ARTICLE.read_text(encoding="utf-8")
print("Отправляем статью в Ollama. Ждём...")

review = generate_review(article_text)

print("\n=== РЕЦЕНЗИЯ (валидированный dict по схеме) ===")
print(json.dumps(review, indent=4, ensure_ascii=False))
print("\nКлючи ответа:", list(review))

## 2. Количественный тест устойчивости

Проверяем два дефекта генерации на 20 случайных статьях: **(а)** валидность JSON и
**(б)** языковой сдвиг (сколько латинских слов просочилось в русский ответ).

In [ ]:
"""Блок 2. Количественный тест устойчивости на случайных статьях архива."""
import csv
import random
import time

from src.config import RAW_ARTICLES_DIR, REPORTS_DIR
from src.ollama_client import generate_review
from src.quality import check_language_consistency

N_ARTICLES = 20
RANDOM_SEED = 42
OUTPUT_CSV = REPORTS_DIR / "batch_reviews.csv"


def sample_articles(root, count, seed=RANDOM_SEED):
    """Случайная, но воспроизводимая выборка статей (seed фиксирован)."""
    paths = sorted(root.rglob("*.txt"))
    if not paths:
        raise FileNotFoundError(f"В {root} не найдено .txt-статей (архив в git не публикуется)")
    return random.Random(seed).sample(paths, min(len(paths), count))


def run_robustness_test():
    if not RAW_ARTICLES_DIR.exists():
        print(f"❌ Каталог {RAW_ARTICLES_DIR} не найден — нужен архив статей")
        return

    REPORTS_DIR.mkdir(parents=True, exist_ok=True)
    articles = sample_articles(RAW_ARTICLES_DIR, N_ARTICLES)

    valid_json = 0
    russian = 0
    processed = 0
    rows = []

    print(f"🚀 Тестируем модель на {len(articles)} случайных статьях...\n")
    for index, path in enumerate(articles, start=1):
        article_text = path.read_text(encoding="utf-8", errors="ignore")
        started = time.time()
        try:
            # schema-constrained decoding: невалидный JSON практически невозможен
            review = generate_review(article_text)
            valid_json += 1
            is_russian, latin_words = check_language_consistency(review)
            russian += int(is_russian)
        except Exception as error:  # сеть/таймаут/невалидный JSON (ValueError)
            print(f"[{index}/{len(articles)}] {path.name}: ❌ {error}")
            continue

        processed += 1
        elapsed = round(time.time() - started, 1)
        rows.append(
            {
                "file": path.name,
                "category": path.parent.name,
                "valid_json": True,
                "russian": is_russian,
                "latin_words": latin_words,
                "methodology": review.get("оценка_методологии"),
                "novelty": review.get("оценка_новизны"),
                "verdict": review.get("вердикт"),
                "seconds": elapsed,
            }
        )
        status = "✅ RU" if is_russian else "⚠️ ENG"
        print(f"[{index}/{len(articles)}] {path.name} | {status} (latin={latin_words}) | {elapsed} c")

    if not processed:
        print("❌ Ни одного успешного ответа — проверьте, что Ollama запущена")
        return

    with OUTPUT_CSV.open("w", encoding="utf-8", newline="") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

    print("\n" + "=" * 55)
    print("📊 РЕЗУЛЬТАТЫ (Robustness Test)")
    print("=" * 55)
    print(f"Обработано статей:               {processed}/{len(articles)}")
    print(f"JSON Validity Rate:              {valid_json / processed * 100:.1f}% ({valid_json}/{processed})")
    print(f"Language Consistency (русский):  {russian / processed * 100:.1f}% ({russian}/{processed})")
    print(f"Среднее время отклика:           {sum(r['seconds'] for r in rows) / processed:.1f} c")
    print(f"CSV с построчными результатами:  {OUTPUT_CSV}")
    print("=" * 55)


run_robustness_test()

## 3. Качественный тест: LLM-as-a-Judge

Судья — DeepSeek-V3.1: оценивает рецензию локальной модели по трём критериям
(точность, полнота, логика). Результат дописывается в `reports/evaluation_results.jsonl`.

In [ ]:
"""Блок 3. Качественный тест: DeepSeek-V3 оценивает рецензии нашей модели (LLM-as-a-Judge)."""
import json
import os

from dotenv import load_dotenv
from openai import AsyncOpenAI

from src.config import RAW_ARTICLES_DIR, REPORTS_DIR
from src.ollama_client import generate_review

load_dotenv()
CHUTES_API_KEY = os.getenv("CHUTES_API_TOKEN")
if not CHUTES_API_KEY:
    raise ValueError("Ключ CHUTES_API_TOKEN не найден — заполните .env (см. .env.example)")

JUDGE_MODEL = "deepseek-ai/DeepSeek-V3.1-TEE"
SAMPLE_SIZE = 5
RANDOM_SEED = 7
OUTPUT_LOG = REPORTS_DIR / "evaluation_results.jsonl"

client = AsyncOpenAI(api_key=CHUTES_API_KEY, base_url="https://llm.chutes.ai/v1")

JUDGE_SYSTEM_PROMPT = (
    "Ты строгий ИИ-судья. Оцени точность, полноту и логику рецензии "
    "по 10-балльной шкале. Отвечай на русском."
)


def build_judge_prompt(article_text: str, review_json: str) -> str:
    """Промпт судьи: три критерия + исходная статья + проверяемая рецензия."""
    return f"""Ты — главный редактор научного журнала и эксперт по оценке ИИ-моделей.
Младший аналитик (другая нейросеть) прочитал научную статью и составил на нее рецензию в формате JSON.
Твоя задача — оценить качество этой рецензии, опираясь на оригинальный текст статьи.

Оцени работу младшего аналитика по 10-балльной шкале по трем критериям:
1. Точность (Accuracy): Не выдумал ли аналитик факты? Верно ли он понял суть статьи?
2. Полнота (Completeness): Выделил ли он действительно важные сильные и слабые стороны, или отделался общими фразами?
3. Логика (Reasoning): Соответствует ли финальный вердикт (Принять/Отклонить) выставленным оценкам и найденным минусам?

Выведи ответ в строгом и четком формате:
ТОЧНОСТЬ: [Оценка/10] - [Краткое обоснование]
ПОЛНОТА: [Оценка/10] - [Краткое обоснование]
ЛОГИКА: [Оценка/10] - [Краткое обоснование]
ОБЩИЙ ВЫВОД: [Краткий итог качества работы модели]

--- ОРИГИНАЛЬНАЯ СТАТЬЯ ---
{article_text}

--- РЕЦЕНЗИЯ МЛАДШЕГО АНАЛИТИКА (QWEN) ---
{review_json}
"""


async def judge_review(article_text: str, review: dict) -> str:
    """Отправляет пару (статья, рецензия) судье и возвращает его вердикт."""
    response = await client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": build_judge_prompt(article_text, json.dumps(review, ensure_ascii=False))},
        ],
        temperature=0.1,
    )
    return response.choices[0].message.content


async def main():
    import random

    paths = sorted(RAW_ARTICLES_DIR.rglob("*.txt"))
    if not paths:
        print(f"❌ Нет статей в {RAW_ARTICLES_DIR} — нужен архив (в git не публикуется)")
        return

    REPORTS_DIR.mkdir(parents=True, exist_ok=True)
    articles = random.Random(RANDOM_SEED).sample(paths, min(len(paths), SAMPLE_SIZE))
    print(f"🎯 Оцениваем качество рецензий на {len(articles)} статьях...\n")

    for index, path in enumerate(articles, start=1):
        print(f"[{index}/{len(articles)}] Файл: {path.name}")
        try:
            article_text = path.read_text(encoding="utf-8", errors="ignore")
            review = generate_review(article_text)            # локальная Qwen2.5-7B-LoRA
            judgment = await judge_review(article_text, review)  # DeepSeek-V3 как судья
            entry = {"file": path.name, "qwen_review": review, "deepseek_judgment": judgment}
            with OUTPUT_LOG.open("a", encoding="utf-8") as log_file:
                log_file.write(json.dumps(entry, ensure_ascii=False) + "\n")
            print(f"✅ Готово. Итог судьи: {judgment.splitlines()[-1]}")
            print("-" * 30)
        except Exception as error:
            print(f"❌ Ошибка на файле {path.name}: {error}")


# В Jupyter можно просто `await main()`; при экспорте в .py — asyncio.run(main())
await main()

## 4. Графики для отчёта

Числа на графиках — из сохранённых прогонов (`reports/`). При повторном запуске
блоков 2–3 подставьте свои значения.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from src.config import FIGURES_DIR

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Настройки стиля
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 12, 'font.family': 'sans-serif'})

def plot_robustness_donut():
    """Создает кольцевую диаграмму для количественного теста (Рисунок 8)"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
    
    # Данные прогона на 20 статьях (reports/ и docs/results.md)
    # График 1: Валидность JSON
    sizes_json = [100, 0]
    colors_json = ['#4C72B0', '#DDDDDD']
    
    # График 2: Языковая консистентность
    sizes_lang = [95, 5]
    colors_lang = ['#55A868', '#C44E52']
    
    # Рисуем кольца
    wedges1, texts1, autotexts1 = ax1.pie(sizes_json, colors=colors_json, autopct='%1.1f%%', 
                                          startangle=90, pctdistance=0.85, 
                                          textprops=dict(color="white", weight="bold", size=14))
    wedges2, texts2, autotexts2 = ax2.pie(sizes_lang, colors=colors_lang, autopct='%1.1f%%', 
                                          startangle=90, pctdistance=0.85,
                                          textprops=dict(color="white", weight="bold", size=14))
    
    # Делаем дырки в центре
    centre_circle1 = plt.Circle((0,0),0.70,fc='white')
    centre_circle2 = plt.Circle((0,0),0.70,fc='white')
    ax1.add_patch(centre_circle1)
    ax2.add_patch(centre_circle2)
    
    ax1.set_title("Валидность JSON\n(JSON Validity Rate)", weight="bold", pad=20)
    ax2.set_title("Языковая целостность\n(Language Consistency)", weight="bold", pad=20)
    
    # Подписи в центре
    ax1.text(0, 0, '20/20\nСтатей', ha='center', va='center', fontsize=14, weight='bold', color='#333333')
    ax2.text(0, 0, '19/20\nСтатей', ha='center', va='center', fontsize=14, weight='bold', color='#333333')
    
    # plt.suptitle("Рисунок 8. Метрики технической надежности генерации (Robustness Test)", fontsize=16, weight="bold", y=1.05)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'robustness_metrics.png', dpi=300, bbox_inches='tight')
    print("✅ График сохранен: robustness_metrics.png")

def plot_llm_judge_radar():
    """Создает лепестковую диаграмму (радар) для качественного теста (Рисунок 9)"""
    # Средние баллы судьи из reports/evaluation_results.jsonl (5 статей)
    categories = ['Точность\n(Accuracy)', 'Полнота\n(Completeness)', 'Логика\n(Reasoning)']
    scores = [7.2, 5.2, 5.2]
    
    # Замыкаем круг для радара
    categories = np.concatenate((categories, [categories[0]]))
    scores = np.concatenate((scores, [scores[0]]))
    
    angles = np.linspace(0, 2 * np.pi, len(categories) - 1, endpoint=False).tolist()
    angles += angles[:1]
    
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    
    # Отрисовка
    ax.plot(angles, scores, color='#4C72B0', linewidth=2.5, linestyle='solid')
    ax.fill(angles, scores, color='#4C72B0', alpha=0.25)
    
    # Настройки осей (от 0 до 10)
    ax.set_ylim(0, 10)
    ax.set_yticks([2, 4, 6, 8, 10])
    ax.set_yticklabels(['2', '4', '6', '8', '10'], color="grey", size=10)
    
    # Подписи категорий
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories[:-1], size=13, weight="bold")
    
    # plt.title("Рисунок 9. Семантическая оценка генерации методом LLM-as-a-Judge\n(Средние баллы из 10)", 
            #   fontsize=15, weight="bold", pad=30)
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'llm_judge_radar.png', dpi=300, bbox_inches='tight')
    print("✅ График сохранен: llm_judge_radar.png")

if __name__ == "__main__":
    plot_robustness_donut()
    plot_llm_judge_radar()